# 03 · rPPG 기준 오차 기반 품질 선택기

기존 POS/CHROM/green · 영역별 후보를 유지하고, **실제 기준 심박(검증된 동기화)** 과 비교했을 때 어떤 후보가 덜 틀리는지를 학습합니다. 기존 confidence 나 엔진 HR 을 정답으로 삼는 모방 학습이 아닙니다.

- 라벨: 같은 세션의 기준 심박(BLE 또는 사후 CSV, `dataset_annotations` 의 최신 유효 연결). 동기화 불확실성 ≤100 ms 로 연구자가 검증한 경우만 `paired`, 그 밖은 `exploratory`(학습·승격 제외, 별도 보고).
- 고정 10초 epoch · 기준 관측률 ≥0.8 · 2.5초 넘는 기준 손실은 잇지 않음(condition-dataset.js 와 동일 규칙). 기준이 있는데 후보가 없는 epoch 는 분모에 남습니다.
- 실시간 배포용은 창 안 정보만 쓴 후보(causal)만 사용합니다. 이웃 창으로 받아들인(tracked) 창의 근거는 offline 전용으로 표시합니다.
- **HR-only 기준으로 IBI/HRV 라벨을 만들지 않습니다.** 박동 의존 지표는 승격 후에도 ‘미검증’으로 남습니다.
- paired 세션 < 6 또는 epoch < 60 이면 `insufficient_reference_labels` 로 끝나고 기존 엔진을 유지합니다. 수집·정렬·QC 경로는 그대로 동작합니다.

In [ ]:
#@title ① 환경
import os, subprocess, sys
if not os.path.isdir('/content/neurolens'): subprocess.check_call(['git', 'clone', '--depth', '50', 'https://github.com/jeebs0627/neurolens.git', '/content/neurolens'])
os.chdir('/content/neurolens'); sys.path.insert(0, '/content/neurolens/tools/colab')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'scipy'])
from google.colab import drive  # type: ignore
drive.mount('/content/drive'); RAW = '/content/drive/MyDrive/neurolens-training/raw'; WORK = '/content/drive/MyDrive/neurolens-training/work'

In [ ]:
#@title ② 기준 pairing → 후보 오차 라벨 → 선택기 학습·평가
from nlcolab.pipeline import Pipeline
import json
pl = Pipeline(RAW, WORK)
res = pl.rppg()
print(json.dumps({k: res[k] for k in ('status', 'pairedSessions', 'pairedEpochs', 'exploratoryRows', 'referenceGrades', 'hrvNote')}, ensure_ascii=False, indent=2))
if res['status'] == 'evaluated':
    print('validation:', json.dumps(res['val'], ensure_ascii=False, indent=2)[:3000])
    print('test (release gate 1회):', json.dumps(res['test'], ensure_ascii=False, indent=2)[:3000])

## 읽는 법
- `modelCoverage` 대 `engineCoverage`, 같은 coverage 에서의 MAE, `riskCoverage`(예상 오차 문턱별 활용률·오차) 를 함께 봅니다. 오류가 커졌는데 출력량만 늘어난 모델은 실패입니다.
- `oracle` 은 ‘후보 중 실제 최선’의 연구용 상한일 뿐 운영 모델이 아닙니다.
- 기준이 `exploratory` 뿐이면 **먼저 동기화 근거를 검증해 paired 로 바꾸는 것**이 다음 과제입니다(`/dataset` → 기준 CSV · 동기화 오차 상한 · 검토 체크).